# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their field @id's
record_sets = list(dataset.metadata.record_sets)
print("Available Record Sets (by @id):")
for rset in record_sets:
    print(f"- {rset['@id']} : {rset.get('name', 'Unnamed')}")

# Show all fields for each record set
for rset in record_sets:
    print(f"\nFields in Record Set @id '{rset['@id']}':")
    for field in rset.get('field', []):
        # If field is a dict with @id, print id and name (if available).
        if isinstance(field, dict):
            fid = field['@id']
            fname = field.get('name', '')
            print(f"  - {fid}{': ' + fname if fname else ''}")
        else:
            print(f"  - {field}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets into DataFrames
dataframes = {}
record_set_ids = [rset['@id'] for rset in record_sets]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Display available fields (columns) for each record set
for rs_id in dataframes:
    print(f"\nRecord Set: {rs_id}\nColumns: {dataframes[rs_id].columns.tolist()}")
    print(dataframes[rs_id].head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select an example record set and perform EDA
# Update these @id's before running if you want to use a different field/record set as per your dataset content
# Here, we try the first available numeric field in the first record set for demo

# Pick the first record set
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# Try to pick numeric fields by pandas dtype
numeric_fields = df.select_dtypes(include=[float, int]).columns.tolist() if not df.empty else []
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field '{numeric_field_id}' from record set '{record_set_id}' for demo analysis.")
else:
    print("No numeric fields found; change the record_set_id/numeric_field_id as appropriate.")

if numeric_fields:
    threshold = df[numeric_field_id].mean() if not pd.isna(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
    print(filtered_df.head())

    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    # Try to group by a categorical field if available
    group_fields = df.select_dtypes(include=["object"]).columns.tolist()
    group_field = group_fields[0] if group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index(name="mean_value")
        print(f"Grouped mean of {numeric_field_id} by '{group_field}':")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualize the distribution of a numeric field
import matplotlib.pyplot as plt

if numeric_fields:
    plt.figure(figsize=(7,4))
    df[numeric_field_id].hist(bins=50)
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {numeric_field_id} in Record Set {record_set_id}")
    plt.show()

    # If a group field was found, show boxplot
    if group_field:
        plt.figure(figsize=(8,5))
        df.boxplot(column=numeric_field_id, by=group_field, grid=False, rot=45)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

The notebook demonstrated the use of the `mlcroissant` library to explore the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset. Using the dataset's Croissant schema, we listed the record sets and fields, loaded raw data, performed basic exploratory data analysis (EDA), and visualized key numeric attributes.

- You can extend this notebook by adjusting the `record_set_id` and field IDs based on the record set/fields most relevant to your use case.
- Try further statistical and visual analyses or link to the dataset documentation for detailed field definitions as appropriate.